# Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
import seaborn as sns
import time
import joblib
import os

# metrics
from sklearn.metrics import make_scorer, mean_absolute_error, median_absolute_error, root_mean_squared_error

# cross-validation
from sklearn.model_selection import train_test_split, cross_validate, cross_val_predict, KFold, RandomizedSearchCV

# preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

# regression models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from tabpfn import TabPFNRegressor

# Load data

In [ ]:
df = pd.read_csv('../data/house_pricing/house_pricing_prepared.csv')
df.head()

In [ ]:
# Only GarageYrBlt and SalePrice have nulls

df.columns[df.isna().any()]

## Split column

For training and leaderboard prediction.

In [ ]:
leaderboard_df = df[df['Split'] == 'leaderboard'].copy()
df = df[df['Split'] == 'labeled']

# Paths

In [ ]:
exercise_name = 'house_pricing'
cv_objects_path = os.path.join('cv', exercise_name)

# Feature Eng. Pipeline

In [ ]:
# Ordinal encoding: order matters (index 0 = lowest).
# HouseStyle and MSZoning orders come from the price-ordering computed in the preparation notebook.
ordinal_categories = {
    "Street": ["Grvl", "Pave"],
    "CentralAir": ["N", "Y"],
    "LandSlope": ["Gtl", "Mod", "Sev"],
    "PavedDrive": ["N", "P", "Y"],
    "GarageFinish": ["NoGarage", "Unf", "RFn", "Fin"],
    "Electrical": ["FuseP", "FuseF", "FuseA", "SBrkr"],
    "HouseStyle": ['1.5Unf', '1.5Fin', 'SFoyer', '2.5Unf', 'SLvl', '1Story', '2.5Fin', '2Story'],
    "MSZoning": ['C (all)', 'RM', 'RH', 'RL', 'FV'],
}

onehot_features = ["RoofStyle", "Foundation", "GarageType"]


In [ ]:
ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=list(ordinal_categories.values()))),
])

onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

garage_yr_pipeline = Pipeline([
    ('impute0', SimpleImputer(strategy='constant', fill_value=0)),
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal', ordinal_pipeline, list(ordinal_categories.keys())),
        ('onehot', onehot_pipeline, onehot_features),
        ('impute0', garage_yr_pipeline, ['GarageYrBlt']),
    ],
    remainder='passthrough',  # deixa la resta de features tal i com estan
)

# Train / Test split
Keep test split totally isolated from cross-validation.

In [ ]:
target_ft = 'SalePrice'
fts2drop = ['Id', 'Split', target_ft]

In [ ]:
X = df.drop(columns=fts2drop)
y = df[target_ft]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42,
)

In [ ]:
cat_cols = X.select_dtypes(include='object').columns
X_train[cat_cols] = X_train[cat_cols].astype('category')
X_test[cat_cols] = X_test[cat_cols].astype('category')

# Cross-validation

In [ ]:
# Define CV
kf = KFold(10, shuffle=True, random_state=42)

In [ ]:
# Define metrics
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'RMSE': make_scorer(root_mean_squared_error),
    'MedAE': make_scorer(median_absolute_error),
}

In [ ]:
# Dict to store CV results for the different models
cv_scores = {}

# Functions

In [ ]:
def store_cv_object(cv_object, fpath, fname):
    """Persist a cross-validation object to disk with joblib.

    Works for any picklable CV result: the dict returned by `cross_validate`
    or a fitted search object such as `RandomizedSearchCV` / `GridSearchCV`.
    joblib is preferred over plain pickle here because these objects hold
    large numpy arrays, which joblib serializes more efficiently.

    Args:
        cv_object: The object to store (a `cross_validate` dict or a fitted
            `*SearchCV` instance).
        fpath: Destination file path (e.g. 'cv/housing/').
        fname: File name (e.g. 'baseline1.joblib').

    Returns:
        The `fpath` it was written to.
    """
    # Create the folder if it does not exist
    os.makedirs(fpath, exist_ok=True)
    full_path = os.path.join(fpath, fname)
    # Store object
    joblib.dump(cv_object, full_path)
    return full_path


def select_best_cv(cv_results):
    # Define what the best model is of a hyperparameter search
    return cv_results["mean_test_MAE"].argmin()


def get_cv_scores_df(model, scoring, index=None):
    """Print train/validation regression scores from cross_validate or a search.

    Args:
        model: Either a dict returned by `cross_validate`, or a fitted
            `*SearchCV` object exposing `cv_results_` and `best_params_`.
        scoring: dict of scorers (its keys name the metrics to report).
        index: Row index into `cv_results_` to report (e.g. `best_index_`).
            Required when `model` is a search object; ignored otherwise.
    Returns:
        DataFrame indexed by metric with 'Train' and 'Validation' columns.
    """
    if hasattr(model, 'cv_results_'):
        print('Best hyperparameters:')
        pprint(model.best_params_, indent=4, width=40)
        get = lambda split, m: model.cv_results_[f'mean_{split}_{m}'][index]
    else:
        get = lambda split, m: model[f'{split}_{m}'].mean()

    return pd.DataFrame(
        {'Train': [get('train', m) for m in scoring],
         'Validation': [get('test', m) for m in scoring]},
        index=list(scoring),
    ).round(1)
    

def real_vs_pred(model, X_train, y_train, kfold):
    """Plot cross-validated predictions against the real target values.

    Predictions are produced with `cross_val_predict` so every point is an
    out-of-fold estimate. A diagonal y = x reference line is drawn: points
    on the line are perfect predictions.

    Args:
        model: An unfitted scikit-learn estimator or pipeline.
        X_train: Feature matrix used for cross-validation.
        y_train: Target vector aligned with `X_train`.

    Note:
        Uses the module-level `kf` as the cross-validation splitter.
    """
    preds = cross_val_predict(model, X_train, y_train, cv=kfold, n_jobs=-1)
    x_line = np.arange(y_train.min(), y_train.max())
    plt.scatter(y_train, preds)
    plt.plot(x_line, x_line, color='orange')
    plt.xlabel('Real target')
    plt.ylabel('Predicted target')
    plt.show()


def create_submission(model, X, y, leaderboard_df, fts2drop, fpath=''):
    """Fit a model on all training data and write leaderboard predictions to CSV.

    Args:
        model: An unfitted scikit-learn estimator or pipeline.
        X: Training feature matrix.
        y: Training target vector aligned with `X`.
        leaderboard_df: Leaderboard set, including an 'Id' column and the
            same raw features as the training data.
        fts2drop: Columns to drop from `leaderboard_df` so its features match
            those the model was trained on (e.g. 'Id' and the target).
        fpath: Output CSV path. Defaults to 'house_pricing_submission.csv'
            when empty.
    """
    X_lb = leaderboard_df.drop(columns=fts2drop)
    model.fit(X, y)
    preds = model.predict(X_lb)
    submission_df = pd.DataFrame({
        'Id': leaderboard_df['Id'],
        'prediction': preds
    })
    if not fpath:
        fpath = 'house_pricing_submission.csv'
    submission_df.to_csv(fpath, index=False)

# Derive Fts (no leakage)

In [ ]:
X["BuiltArea"].describe()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class PriceByAreaBinTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y):
        # Define bins from training data
        self.bins_ = np.arange(X["BuiltArea"].min(), X["BuiltArea"].max() + 500, 500)
        self.labels_ = range(len(self.bins_) - 1)

        # Store the global mean for unseen bins
        self.global_mean_ = float(np.mean(y))

        # Compute mean SalePrice per bin
        area_bin = pd.cut(X["BuiltArea"], bins=self.bins_, labels=self.labels_)
        df_temp = pd.DataFrame({"AreaBin": area_bin, "SalePrice": y})
        self.bin_means_ = (
            df_temp.groupby("AreaBin", observed=True)["SalePrice"]
            .mean()
            .to_dict()
        )
        return self

    def transform(self, X):
        X_ = X.copy()
        area_bin = pd.cut(X_["BuiltArea"], bins=self.bins_, labels=self.labels_)
        # Map and explicitly cast to float
        mapped = area_bin.map(self.bin_means_).astype(float)
        X_["SalePrice_by_AreaBin"] = mapped.fillna(self.global_mean_)
        return X_

# Modeling

## Baseline 1

In [ ]:
baseline1 = DummyRegressor(strategy='mean')

baseline1_cv = cross_validate(
    baseline1,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# Store CV
store_cv_object(baseline1_cv, cv_objects_path, 'baseline1.joblib')

# Print CV metrics
cv_scores['baseline1'] = get_cv_scores_df(baseline1_cv, scoring)
cv_scores['baseline1']

## Baseline 2

Linear regression with a single feature that looked important `GrLivArea`.

In [ ]:
baseline2 = LinearRegression(n_jobs=-1)

baseline2_cv = cross_validate(
    baseline2,
    X_train[['GrLivArea']],
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# Store CV
store_cv_object(baseline2_cv, cv_objects_path, 'baseline2.joblib')

# Print CV metrics
cv_scores['baseline2'] = get_cv_scores_df(baseline2_cv, scoring)
cv_scores['baseline2']

## EN

In [ ]:
en_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', None),
    ('en', ElasticNet(max_iter=10_000))
])

param_dist = {
    'scaler': [StandardScaler(), RobustScaler(), MinMaxScaler()],
    'en__l1_ratio': np.arange(0, 1.01, 0.01),
    'en__alpha': np.arange(0.01, 1.01, 0.01),
}

# Entrenar
en_rs = RandomizedSearchCV(
    en_pipeline,
    param_distributions=param_dist,
    n_iter=50,
    scoring=scoring,
    return_train_score=True,
    cv=kf,
    refit=select_best_cv,
    random_state=42,
    n_jobs=-1
)
en_rs.fit(X_train, y_train)

print('Best RandomizedSearchCV parameters: ', en_rs.best_params_)

# Store CV
store_cv_object(en_rs, cv_objects_path, 'en.joblib')

# Print CV metrics
cv_scores['en'] = get_cv_scores_df(en_rs, scoring, en_rs.best_index_)
cv_scores['en']

In [ ]:
# real vs predicted
real_vs_pred(en_pipeline, X_train, y_train, kf)

## RF

### Hyperparameter tunning

In [ ]:
rf = Pipeline([
    ('feat', PriceByAreaBinTransformer()),
    ('rf', RandomForestRegressor(n_jobs=-1, random_state=42)),
])

params = [{
    'rf__n_estimators': [100, 200, 300],
    'rf__criterion': ['friedman_mse', 'squared_error', 'absolute_error'],
    'rf__min_samples_split': [2, 5, 10, 20],
    'rf__min_samples_leaf': [1, 2, 4, 6],
    'rf__max_depth': [3, 4, 5, 6, 8, 10, 20, 50],
    'rf__max_features': [None, 'sqrt', 'log2', 0.5, 0.7, 0.9],
    'rf__max_samples': [None, 0.5, 0.7, 0.9],
    'rf__max_leaf_nodes': [None, 5, 10, 20],
    'rf__min_weight_fraction_leaf': np.arange(0, 0.51, 0.1),
}]

rf_rs = RandomizedSearchCV(
    rf,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit=select_best_cv,
    return_train_score=True,
    n_jobs=-1
)

rf_rs.fit(X_train, y_train)

# Store CV
store_cv_object(rf_rs, cv_objects_path, 'rf.joblib')

# Print CV metrics
cv_scores['rf'] = get_cv_scores_df(rf_rs, scoring, rf_rs.best_index_)
cv_scores['rf']

In [ ]:
# Get the indices of the smallest MAE values
best_indices = np.argsort(rf_rs.cv_results_['mean_test_MAE'])[:5]

# Show best MAEs and their corresponding hyperparams.
for i in best_indices:
    print('Validation MAE:', rf_rs.cv_results_['mean_test_MAE'][i].round(2))
    print('Hyperparams:')
    pprint(rf_rs.cv_results_['params'][i], indent=4, width=40)
    print()

In [ ]:
# real vs predicted
real_vs_pred(rf_rs.best_estimator_, X_train, y_train, kf)

### Feature importances

In [ ]:
rf_rs.best_estimator_.fit(X_train, y_train)

rf_ft_imps = pd.DataFrame({
    'feature': list(X_train.columns) + ['SalePrice_by_AreaBin'],
    'importance': rf_rs.best_estimator_['rf'].feature_importances_
}).sort_values('importance', ascending=False).round(3)

rf_ft_imps.head(20)

## HGBDT

### Round 1

In [ ]:
gb = Pipeline([
    ('feat', PriceByAreaBinTransformer()),
    ('gb', HistGradientBoostingRegressor(random_state=42)),
])

params = {
    'gb__loss': ['squared_error', 'absolute_error'],
    'gb__learning_rate': np.logspace(-3, 0, 10),
    'gb__max_iter': [20, 50, 100, 200, 300, 500, 1000],
    'gb__max_leaf_nodes': [15, 31, 63, 127], 
    'gb__max_depth': [None, 3, 5, 8, 12, 20],
    'gb__min_samples_leaf': [10, 20, 40, 80],
    'gb__max_features': [0.5, 0.7, 0.9, 1.0], # Randomly samples x% of columns at each split
    'gb__max_bins': [63, 127, 255],  # Feature bucket resolution (max 255)
    'gb__l2_regularization': [0.0, 0.1, 1.0], # Helps prevent overfitting
    'gb__validation_fraction': [0.1, 0.2],
}

gb_rs = RandomizedSearchCV(
    gb,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit=select_best_cv,
    return_train_score=True,
    n_jobs=-1
)

gb_rs.fit(X_train, y_train)

# Store CV
store_cv_object(gb_rs, cv_objects_path, 'gb.joblib')

# Print CV metrics
cv_scores['gb'] = get_cv_scores_df(gb_rs, scoring, gb_rs.best_index_)
cv_scores['gb']

In [ ]:
# Get the indices of the smallest MAE values
best_indices = np.argsort(gb_rs.cv_results_['mean_test_MAE'])[:5]

# Show best MAEs and their corresponding hyperparams.
for i in best_indices:
    print('Validation MAE:', gb_rs.cv_results_['mean_test_MAE'][i].round(2))
    print('Hyperparams:')
    pprint(gb_rs.cv_results_['params'][i], indent=4, width=40)
    print()

### Round 2

In [ ]:
gb = Pipeline([
    ('feat', PriceByAreaBinTransformer()),
    ('gb', HistGradientBoostingRegressor(
        learning_rate=0.02154,
        min_samples_leaf=10,
        random_state=42,
    )),
])

params = {
    'gb__loss': ['squared_error', 'absolute_error'],
    'gb__max_depth': range(3, 20),
    'gb__max_iter': range(100, 600, 100),
    'gb__max_leaf_nodes': [15, 127], 
    'gb__max_features': np.arange(0.4, 0.8, 0.05),
    'gb__l2_regularization': [0.0, 0.1, 1.0],
    'gb__max_bins': [63, 127, 255],
    'gb__validation_fraction': [0.1, 0.2],
}

gb_rs = RandomizedSearchCV(
    gb,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit=select_best_cv,
    return_train_score=True,
    n_jobs=-1
)

gb_rs.fit(X_train, y_train)

# Store CV
store_cv_object(gb_rs, cv_objects_path, 'gb.joblib')

# Print CV metrics
cv_scores['gb'] = get_cv_scores_df(gb_rs, scoring, gb_rs.best_index_)
cv_scores['gb']

In [ ]:
# real vs predicted
real_vs_pred(gb_rs.best_estimator_, X_train, y_train, kf)

In [ ]:
# Boxplot with validation MAE

y_val_pred = cross_val_predict(gb_rs.best_estimator_, X_train, y_train, cv=kf, n_jobs=-1)

plt.boxplot(abs(y_train - y_val_pred))
plt.ylabel('Validation MAE')
plt.yscale('log')
plt.show()

## XGBoost

In [ ]:
from xgboost import XGBRegressor

In [ ]:

# Define the pipeline with XGBRegressor
xgb_pipeline = Pipeline([
    ('feat', PriceByAreaBinTransformer()),
    ('xgb', XGBRegressor(
        random_state=42,
        enable_categorical=True,
        tree_method='hist',
    )),
])

# XGBoost specific parameters
params = {
    'xgb__objective': ['reg:squarederror', 'reg:absoluteerror'],
    'xgb__learning_rate': np.logspace(-3, 0, 10),
    'xgb__n_estimators': [20, 50, 100, 200, 300, 500, 800, 1000],
    'xgb__max_leaves': range(2, 51), 
    'xgb__max_depth': range(3, 51), # XGBoost defaults to 6, doesn't use None
    'xgb__min_child_weight': range(2, 51), # Equivalent to min_samples_leaf
    'xgb__colsample_bytree': np.arange(0.3, 1.1, 0.1), # Equivalent to max_features
    'xgb__reg_lambda': np.arange(0.4, 1.1, 0.1), # L2 regularization
    'xgb__subsample': np.arange(0.4, 1.1, 0.1), # Corresponds to validation_fraction/sampling
}

# Randomized Search
xgb_rs = RandomizedSearchCV(
    xgb_pipeline,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit=select_best_cv,
    return_train_score=True,
    n_jobs=-1
)

xgb_rs.fit(X_train, y_train)

# Store CV
store_cv_object(xgb_rs, cv_objects_path, 'xgb.joblib')

# Print CV metrics
cv_scores['xgb'] = get_cv_scores_df(xgb_rs, scoring, xgb_rs.best_index_)
cv_scores['xgb']

In [ ]:
# Get the indices of the smallest MAE values
best_indices = np.argsort(xgb_rs.cv_results_['mean_test_MAE'])[:5]

# Show best MAEs and their corresponding hyperparams.
for i in best_indices:
    print('Validation MAE:', xgb_rs.cv_results_['mean_test_MAE'][i].round(2))
    print('Hyperparams:')
    pprint(xgb_rs.cv_results_['params'][i], indent=4, width=40)
    print()

In [ ]:
# real vs predicted
real_vs_pred(xgb_rs.best_estimator_, X_train, y_train, kf)

## TabPFN

In [ ]:
import warnings
import os
os.environ['TABPFN_ALLOW_CPU_LARGE_DATASET'] = '1'

# Suppress the specific warning(s) emitted by the tabpfn library
warnings.filterwarnings('ignore', category=UserWarning, module='tabpfn')

In [ ]:
tabpfn_pipeline = Pipeline([
    ('feat', PriceByAreaBinTransformer()),
    ('tabpfn', TabPFNRegressor(
        model_path='models/tabpfn_v3/tabpfn-v3-regressor-v3_20260417_mediumdata.ckpt',
        device='cpu'
    )),
])

# Perform cross-validation (no need of hyperparameter tuning)
tabpfn_cv = cross_validate(
    tabpfn_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# Store CV
store_cv_object(tabpfn_cv, cv_objects_path, 'tabpfn.joblib')

# Print CV metrics
cv_scores['tabpfn'] = get_cv_scores_df(tabpfn_cv, scoring)
cv_scores['tabpfn']

In [ ]:
# real vs predicted
real_vs_pred(tabpfn_local, X_train, y_train, kf)

## Stacking

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

In [ ]:
xgb_pipeline = Pipeline([
    ('feat', PriceByAreaBinTransformer()),
    ('xgb', XGBRegressor(**{
        'colsample_bytree': np.float64(0.6),
        'learning_rate': np.float64(0.021544346900318832),
        'max_depth': 3,
        'max_leaves': 31,
        'min_child_weight': 5,
        'n_estimators': 1000,
        'objective': 'reg:squarederror',
        'reg_lambda': np.float64(0.5),
        'subsample': np.float64(0.4),
        'enable_categorical': True,
        'tree_method': 'hist',
    }))
])

tabpfn_pipeline = Pipeline([
    ('feat', PriceByAreaBinTransformer()),
    ('tabpfn', TabPFNRegressor(
        model_path='models/tabpfn_v3/tabpfn-v3-regressor-v3_20260417_mediumdata.ckpt',
        device='cpu'
    )),
])

# Force the initialization
# This dummy fit tells TabPFN to load the weights and setup the internal state
# If not we'll have problems when stacking because of some logic there
tabpfn_pipeline.fit(X_train[:2], y_train[:2])

base_models = [
    ('xgb', xgb_pipeline),
    ('tabpfn', tabpfn_pipeline),
]

In [ ]:
stacking = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(),
    cv=kf,
    n_jobs=-1
)

stacking.fit(X_train, y_train)

# Summary

In [ ]:
cv_objs = {
    'BLN 1': baseline1_cv,
    'BLN 2': baseline2_cv,
    'EN': en_rs,
    'RF': rf_rs,
    'GBDT': gb_rs,
    'TabPFN': tabpfn_cv,
}

## Validation metrics

In [ ]:
y_valid_axis = {score_name: [] for score_name in scoring}

for model_name, cv_obj in cv_objs.items():
    if hasattr(cv_obj, 'cv_results_'):
        get = lambda split, m: cv_obj.cv_results_[f'mean_{split}_{m}'][cv_obj.best_index_]
    else:
        get = lambda split, m: cv_obj[f'{split}_{m}'].mean()

    for score_name in scoring:
        y_valid_axis[score_name].append(get('test', score_name))

x_axis = np.arange(len(cv_objs))
width = 0.4 / (len(y_valid_axis) / 2) # Automatically scales down if you add more bars
total_bars = len(y_valid_axis)

# Grab a built-in color palette
cmap = plt.get_cmap('Set2')

# Plot bars for each metric
for i, (label, y_values) in enumerate(y_valid_axis.items()):
    # Formula to center-align any number of bars around the x-axis tick
    offset = (i - (total_bars - 1) / 2) * width
    plt.bar(
        x_axis + offset,
        y_values,
        width,
        label=label,
        color=cmap(i),
    )

plt.xticks(x_axis, cv_objs.keys())
plt.ylabel('Error')
plt.title('Validation Metrics')
plt.legend()
plt.show()

In [ ]:
pd.DataFrame(y_valid_axis, index=models.keys()).round(1).T

## Train vs Validation

In [ ]:
y_train_axis = []
y_valid_axis = []

for name, model in models.items():
    if hasattr(model, 'cv_results_'):
        get = lambda split, m: model.cv_results_[f'mean_{split}_{m}'][model.best_index_]
    else:
        get = lambda split, m: model[f'{split}_{m}'].mean()

    y_train_axis.append(get('train', 'MAE'))
    y_valid_axis.append(get('test', 'MAE'))

x_axis = np.arange(len(models))
plt.bar(x_axis - 0.2, y_train_axis, 0.4, label = 'Train')
plt.bar(x_axis + 0.2, y_valid_axis, 0.4, label = 'Validation')

plt.xticks(x_axis, models.keys())
plt.ylabel('MAE')
plt.title('Train vs Validation MAE')
plt.legend()
plt.show()

# Test

Choose the most promising models and test them.

## Evaluation

In [ ]:
models = {
    'XGB': xgb_pipeline,
    'TabPFN': tabpfn_pipeline,
    'Stacking': stacking,
}

In [ ]:
# # Get the list of categories seen during training
# known_categories = list(X_train['Electrical'].unique())

# # Map values in test data to 'Unknown' if they aren't in training
# X_test['Electrical'] = X_test['Electrical'].apply(
#     lambda x: x if x in known_categories else 'Unknown'
# )

In [ ]:
# Compute test predictions for each model

y_test_preds = {}
y_test_axis = {score_name: [] for score_name in scoring}

for name, model in models.items():
    print(name)
    
    # Fit on entire training set
    if name in ('XGB', 'TabPFN'):
        # The other models are already fit with full train set
        model.fit(X_train, y_train)
    y_test_preds[name] = model.predict(X_test)

    # Compute all scores on real test values and predictions
    for score_name in scoring:
        value = scoring[score_name]._score_func(y_test, y_test_preds[name])
        y_test_axis[score_name].append(value)

In [ ]:
# Display test metrics

x_axis = np.arange(len(models))
width = 0.4 / (len(y_test_axis) / 2) # Automatically scales down if you add more bars
total_bars = len(y_test_axis)

# Grab a built-in color palette
cmap = plt.get_cmap('Set2')

# Plot bars for each metric
for i, (label, y_values) in enumerate(y_test_axis.items()):
    # Formula to center-align any number of bars around the x-axis tick
    offset = (i - (total_bars - 1) / 2) * width
    plt.bar(
        x_axis + offset,
        y_values,
        width,
        label=label,
        color=cmap(i),
    )
  
plt.xticks(x_axis, models.keys())
plt.ylabel('MAE')
plt.title('Models')
plt.legend()
plt.show()

In [ ]:
pd.DataFrame(y_test_axis, index=models.keys()).round(1).T

# Submission

In [ ]:
create_submission(
    tabpfn_pipeline,
    X,
    y,
    leaderboard_df,
    fts2drop,
    '../data/house_pricing/house_pricing_tabpfn_submission.csv'
)

# Save Model

Saving the model as a `pickle`, meaning that the Python's object (which is a trained model) is stored as a file in the specified path.

`gb_rs` is an object of `RandomizedSearchCV` type. Out of all the hyperparameters it tried, it stored the best model in the `best_estimator_` attribute.

Then, `gb_rs.best_estimator_` is a `GradientBoostingRegressor` object, which is trained with all training data (X_train and y_train), as you can see in the cell above.

If we think our model is robust enough, one good option is to train it with all the available data we have. However, the absence of additional data to test the model introduces the risk of deploying an unvalidated model to production. To mitigate this risk, we can employ thorough cross-validation on the complete dataset. By using a big number of splits, we ensure a more exhaustive assessment of the model's performance. If the model consistently performs well across numerous cross-validation folds, we can be more assured that expanding the entire dataset will not lead to adverse outcomes in a production environment. This approach does not entirely eliminate the risk, but it significantly bolsters our confidence in the model's generalizability.

In [ ]:
import joblib

In [ ]:
type(gb_rs)

In [ ]:
type(gb_rs.best_estimator_)

In [ ]:
final_cv = cross_validate(
    gb_rs.best_estimator_,
    X,
    y,
    cv=200,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)
print_reg_scores(final_cv)

As we get a good average performance in the cross-validation with all the data, we decide to finally train our model with the whole dataset.

In [ ]:
gb_rs.best_estimator_.fit(X, y)

And store it in a file:

In [ ]:
joblib.dump(gb_rs.best_estimator_, 'models/gbdt.pkl')

# Load Model

We can load and use a trained model stored in a `pickle` file.

`gb` is now a `GradientBoostingRegressor` object that is already trained with X_train and y_train, as this is the model that was previously stored.

We can load and use this object for predicting sale prices of any house data we want, as long as the features are the same as the trained model.

In [ ]:
gb = joblib.load('models/gbdt.pkl')

In [ ]:
type(gb)

Given any dataset of unknown prices, we can now predict its prices (imagine we don't know `X_tests` prices):

In [ ]:
X_unknown = X_test.sample(10)
X_unknown

In [ ]:
# Predicted prices
gb.predict(X_unknown)

We can predict as many instances as we want. For example, a single instance:

In [ ]:
# Single series element
x_single = X_test.sample(1)
x_single

In [ ]:
# Predicted price
gb.predict(x_single)